# Ingest conference sessions into Elasticsearch

Downloads [sessions.json](https://www.ai.engineer/europe/sessions.json) (JSON schedule for [AI Engineer Europe](https://www.ai.engineer/europe/schedule)), then indexes **one vector document per session** in Elasticsearch index `conference_schedule`.

Re-running the ingest cell **drops and recreates** the index.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

ES_URL = os.getenv("ES_URL")
ES_USERNAME = os.getenv("ES_USERNAME")
ES_PASSWORD = os.getenv("ES_PASSWORD")
JINA_API_KEY = os.getenv("JINA_API_KEY")

for name, val in [
    ("ES_URL", ES_URL),
    ("ES_USERNAME", ES_USERNAME),
    ("ES_PASSWORD", ES_PASSWORD),
    ("JINA_API_KEY", JINA_API_KEY),
]:
    if not val:
        raise ValueError(f"Missing {name} in .env")

INDEX_NAME = "conference_schedule"

Embeddings use [Jina Embeddings](https://docs.langchain.com/oss/python/integrations/embeddings/jina) (`jina-embeddings-v2-base-en`), same as the other notebooks. Add `JINA_API_KEY` in `.env` if needed ([Jina dashboard](https://jina.ai/api-dashboard/embedding)).

In [3]:
from langchain_community.embeddings import JinaEmbeddings

embeddings = JinaEmbeddings(
    jina_api_key=JINA_API_KEY,
    model_name="jina-embeddings-v5-text-small"
)

/Users/leonie/Documents/code/agentic_search_demo/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


## Download `sessions.json`

In [4]:
import json
from pathlib import Path
from urllib.request import urlretrieve

SESSIONS_URL = "https://www.ai.engineer/europe/sessions.json"
SESSIONS_PATH = Path("content") / "sessions.json"
SESSIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

urlretrieve(SESSIONS_URL, SESSIONS_PATH)
print(f"Saved to {SESSIONS_PATH.resolve()}")

payload = json.loads(SESSIONS_PATH.read_text(encoding="utf-8"))
sessions = payload["sessions"]
print(f"{len(sessions)} sessions loaded")

Saved to /Users/leonie/Documents/code/agentic_search_demo/data/sessions.json
162 sessions loaded


## Export one markdown file per session (`data/by_type/`)

Files are grouped by session `type` (e.g. `workshop`, `talk`, `keynote`). Each file is **markdown** with a `.txt` extension as in the layout below. Re-running **replaces** the whole `data/by_type` tree.

```
data/by_type/
  workshop/
    Some Title.txt
  talk/
    ...
```

In [5]:
import re
import shutil
from pathlib import Path

BY_TYPE_ROOT = Path("content") / "session_data"


def session_display_title(session: dict) -> str:
    t = (session.get("title") or "").strip()
    if t:
        return t
    parts = [session.get("day"), session.get("time"), session.get("room")]
    parts = [p for p in parts if p]
    return " / ".join(parts) if parts else "untitled_session"


def safe_filename(title: str, max_len: int = 120) -> str:
    base = re.sub(r'[\\/:*?"<>|]', "", title)
    base = re.sub(r"\s+", " ", base).strip() or "untitled_session"
    if len(base) > max_len:
        base = base[:max_len].rstrip()
    return base


def session_to_markdown(session: dict) -> str:
    title = session_display_title(session)
    stype = session.get("type") or "unknown"
    speakers = session.get("speakers") or []
    if not isinstance(speakers, list):
        speakers = [speakers]
    lines = [
        f"# {title}",
        "",
        f"- **Day:** {session.get('day') or '—'}",
        f"- **Time:** {session.get('time') or '—'}",
        f"- **Room:** {session.get('room') or '—'}",
        f"- **Type:** {stype}",
    ]
    if session.get("track"):
        lines.append(f"- **Track:** {session['track']}")
    if speakers:
        lines.append(f"- **Speakers:** {', '.join(str(s) for s in speakers)}")
    lines.append("**Description:**")
    desc = (session.get("description") or "").strip()
    if desc:
        lines.append(desc)
    return "\n".join(lines)


def write_sessions_markdown_by_type(sessions: list, root: Path = BY_TYPE_ROOT) -> None:
    if root.exists():
        shutil.rmtree(root)
    root.mkdir(parents=True, exist_ok=True)
    counts: dict[tuple[str, str], int] = {}

    for session in sessions:
        stype = (session.get("type") or "unknown").strip() or "unknown"
        type_dir = root / stype
        type_dir.mkdir(parents=True, exist_ok=True)
        base = safe_filename(session_display_title(session))
        key = (stype, base)
        n = counts.get(key, 0)
        counts[key] = n + 1
        name = base if n == 0 else f"{base}__{n + 1}"
        path = type_dir / f"{name}.txt"
        path.write_text(session_to_markdown(session), encoding="utf-8")

    print(f"Wrote {len(sessions)} files under {root.resolve()}")


write_sessions_markdown_by_type(sessions)

Wrote 162 files under /Users/leonie/Documents/code/agentic_search_demo/data/session_data


## Build LangChain documents (one per session)

`page_content` is what gets embedded; metadata holds filterable fields.

### What you see in Elasticsearch / Kibana

LangChain’s `ElasticsearchStore` stores each document roughly as:

- **`text`** — the same string as LangChain’s **`page_content`**: **title** plus **description** (when present). Schedule details (day, room, …) live in **`metadata`** only. In Kibana Discover, this is the main body field.
- **`vector`** — the **embedding**: a numeric array produced by Jina from **`text`**. Similarity search compares your query’s embedding to these vectors. You typically do not read `vector` by hand.
- **`metadata`** — structured fields copied from `Document.metadata` (title, day, room, etc.). They are **not** what the embedding model sees unless you also duplicated them inside `page_content`.

In [6]:
import hashlib
from langchain_core.documents import Document

def session_stable_id(session: dict) -> str:
    canonical = json.dumps(session, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

def session_to_document(session: dict) -> Document:
    title = (session.get("title") or "").strip() or "(untitled session)"
    desc = (session.get("description") or "").strip()
    speakers = session.get("speakers") or []
    if not isinstance(speakers, list):
        speakers = [speakers]
    speaker_str = ", ".join(str(s) for s in speakers)
    day = session.get("day") or ""
    time = session.get("time") or ""
    room = session.get("room") or ""
    stype = session.get("type") or ""
    track = session.get("track") or ""

    page_content = title if not desc else f"{title}\n\n{desc}"

    return Document(
        page_content=page_content,
        metadata={
            "title": title,
            "day": day,
            "time": time,
            "room": room,
            "type": stype,
            "track": track,
            "speakers": speaker_str,
        },
    )

documents = [session_to_document(s) for s in sessions]
ids = [session_stable_id(s) for s in sessions]
print(f"Prepared {len(documents)} documents")

Prepared 162 documents


## Drop index (if present) and ingest vectors

Uses the same Elasticsearch connection variables as `00_prepare_context_sources.ipynb` (`ES_URL`, `ES_USERNAME`, `ES_PASSWORD`).

In [7]:
from elasticsearch import Elasticsearch
from langchain_elasticsearch import ElasticsearchStore

es_client = Elasticsearch(ES_URL, basic_auth=(ES_USERNAME, ES_PASSWORD))
if es_client.indices.exists(index=INDEX_NAME):
    es_client.indices.delete(index=INDEX_NAME)
    print(f"Deleted index {INDEX_NAME!r}")

vector_store = ElasticsearchStore(
    es_url=ES_URL,
    es_user=ES_USERNAME,
    es_password=ES_PASSWORD,
    index_name=INDEX_NAME,
    embedding=embeddings,
)

vector_store.add_documents(documents=documents, ids=ids)
print(f"Indexed {len(documents)} sessions into {INDEX_NAME!r}")

Deleted index 'conference_schedule'


/Users/leonie/Documents/code/agentic_search_demo/.venv/lib/python3.14/site-packages/langchain_elasticsearch/_sync/vectorstores.py:692: ElasticsearchWarning: Your license will expire in [4] days. Contact your administrator or update your license for continued use of features
  return self._store.add_texts(


Indexed 162 sessions into 'conference_schedule'


You should now be able to view your data in Kibana under http://127.0.0.1:5601

## Smoke test

In [8]:
hits = vector_store.similarity_search("retrieval", k=3)
for doc in hits:
    print("---")
    print(doc.metadata.get("title"), "|", doc.metadata.get("day"), doc.metadata.get("time"))
    print(doc.page_content[:400] + ("…" if len(doc.page_content) > 400 else ""))

---
Rebranding Retrieval: From RAG over Agentic Retrieval to Agent Memory | April 8 9:00-10:20am
Rebranding Retrieval: From RAG over Agentic Retrieval to Agent Memory
---
Agents on the Canvas in tldraw | April 9 12:40-1:00pm
Agents on the Canvas in tldraw

At tldraw, we've been bringing agents to our infinite canvas. In December 2025, we ran a one-month experiment named Fairydraw where users could work with three fairies—virtual collaborators who work with you, with your human collaborators, and coordinate together on large tasks. Learn what we learned.
---
Connecting the Dots with Context Graphs | April 9 2:30-2:50pm
Connecting the Dots with Context Graphs

AI systems need more than intelligence; they need context that persists. Without it, even strong models can misinterpret information, lose decision rationale, or repeat the same mistakes. Context Graphs have emerged as a practical pattern for agentic AI: a living graph that captures not only what was retrieved or known, but how con